# 07 · Grouped statistics and demeaning

**Module:** 2 · **Time:** 40 minutes · **Prerequisites:** Cleaning, creating variables, merging

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ningzhanglab/Python-for-Beginners/blob/main/modules/02_data_wrangling/07_groupby_aggregate.ipynb) · [![Launch Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/ningzhanglab/Python-for-Beginners/main?filepath=modules%2F02_data_wrangling%2F07_groupby_aggregate.ipynb)

**In Colab choose File → Save a copy in Drive before editing.** These badges link to the currently private repository's `main` after a future merge.

## Learning goals

- **S**ummarize fictional wages by region and employment.
- **C**ompare groupby aggregate and transform operations.
- **C**alculate a transparent weighted group mean.

In [ ]:
import numpy as np
import pandas as pd

survey = pd.DataFrame({
    "person_id": [1001, 1002, 1003, 1003, 1005, 1006],
    "household_id": ["H001", "H001", "H002", "H002", "H003", "H003"],
    "region": [" North ", "SOUTH", "east", "north", "West", "SOUTH"],
    "employment": ["employed", "unemployed", "employed", "employed", "employed", "unemployed"],
    "wage_2023": ["40000", "-99", "51000", "52000", "72000", "not reported"],
    "wage_2024": ["42000", "28000", "53000", "54000", "9999999", "30000"],
    "survey_weight": [1.2, 0.8, 1.1, 1.4, 0.9, 1.6],
})
print("Synthetic teaching extract:", survey.shape)

## Why this matters

Descriptive research frequently compares wages across groups and computes deviations from group means. `groupby().agg()` reduces rows, whereas `transform()` returns one value per original observation. Weighting requires a clear denominator and consistent handling of missing values.

## Prepare a clean numeric column

Clean text and numeric strings first; in this toy example values above 500,000 are flagged and excluded from wage summaries.

In [ ]:
clean = survey.assign(region_clean=lambda d: d["region"].str.strip().str.lower())
raw_wage = pd.to_numeric(clean["wage_2024"], errors="coerce")
clean["wage24"] = raw_wage.mask(raw_wage.gt(500_000))
print(clean[["region_clean", "wage24"]])

## Aggregate to one row per group

`agg` names each output and clarifies which operation it performs. This table combines counts with means; counts of nonmissing wages differ from total rows.

In [ ]:
region_table = clean.groupby("region_clean").agg(
    n_rows=("person_id", "size"),
    n_wages=("wage24", "count"),
    mean_wage=("wage24", "mean")
)
print(region_table)

## Transform keeps the original row count

Within-region demeaned wages are useful when understanding fixed effects. The group mean for each row comes from `transform` rather than a separate merge.

In [ ]:
clean["region_mean"] = clean.groupby("region_clean")["wage24"].transform("mean")
clean["demeaned_wage"] = clean["wage24"] - clean["region_mean"]
assert len(clean) == len(survey)
print(clean[["region_clean", "demeaned_wage"]])

## Weighted group mean

Weights here are illustrative, not a complex survey-design estimator. Exclude missing wages from *both* numerator and denominator.

In [ ]:
valid = clean.loc[clean["wage24"].notna()].copy()
valid["weighted_wage"] = valid["wage24"] * valid["survey_weight"]
weighted = valid.groupby("region_clean").agg(
    numerator=("weighted_wage", "sum"),
    denominator=("survey_weight", "sum")
)
weighted["mean"] = weighted["numerator"] / weighted["denominator"]
print(weighted[["mean"]])

## If you know Stata/R

| Task | pandas | Stata | R |
| --- | --- | --- | --- |
| Group mean | `df.groupby('g')['y'].mean()` | `collapse (mean) y, by(g)` | `group_by(g) |> summarise(mean=mean(y))` |
| Within-group mean | `groupby('g')['y'].transform('mean')` | `bysort g: egen m=mean(y)` | `group_by(g) |> mutate(m=mean(y))` |
| Weighted mean | `sum(w*y) / sum(w)` on valid rows | `summarize y [aw=w]` (weights differ by type) | `weighted.mean(y, w)` |

These weights are classroom numbers; they do not constitute a valid complex-survey variance estimator.

## Try it

Fill each answer cell. In the learner edition, incomplete answers print hints without stopping notebook execution.

### ★ Exercise 1 — Count respondents by group

Set `region_counts` to a Series counting observations in each cleaned region.

In [ ]:
region_counts = None

In [ ]:
expected = clean.groupby("region_clean").size()
if isinstance(region_counts, pd.Series) and region_counts.equals(expected):
    print("✓ Group counts correct")
else:
    print("Hint: use groupby(...).size().")

### ★★ Exercise 2 — Compute mean wages

Create `region_means` with each region's cleaned mean wage.

In [ ]:
region_means = None

In [ ]:
expected = clean.groupby("region_clean")["wage24"].mean()
if isinstance(region_means, pd.Series) and region_means.equals(expected):
    print("✓ Regional means correct")
else:
    print("Hint: group by region_clean and take mean of wage24.")

### ★★ Exercise 3 — Within-group demeaned wages

Set `demeaned` to the Series of wages minus their region-specific means.

In [ ]:
demeaned = None

In [ ]:
expected = clean["wage24"] - clean.groupby("region_clean")["wage24"].transform("mean")
if isinstance(demeaned, pd.Series) and np.allclose(demeaned, expected, equal_nan=True):
    print("✓ Demeaning correct")
else:
    print("Hint: subtract a groupby(...).transform('mean') result.")

### ★★★ Exercise 4 — Weighted average of valid wages

Compute `weighted_overall` from nonmissing wage24 values, using survey_weight for numerator and denominator.

In [ ]:
weighted_overall = None

In [ ]:
expected = np.average(valid["wage24"], weights=valid["survey_weight"])
if isinstance(weighted_overall, (float, int, np.number)) and np.isclose(weighted_overall, expected):
    print("✓ Weighted average correct")
else:
    print("Hint: use np.average with weights from valid rows only.")

## Common errors

- Using `.mean()` on string wage columns gives invalid results; convert wages first.
- A zero-weight denominator makes a weighted mean undefined; validate weights.
- `groupby().agg()` changes row count while `transform()` preserves it.
- Dropping missing wages from the numerator but not from the denominator biases a weighted mean.

## Recap

- Aggregate summarizes groups; transform returns aligned group statistics per row.
- Preserve and report missing-data counts when making summary tables.
- A weighted average requires a consistent valid-sample denominator.
- Within-group demeaning is a useful building block for fixed-effects intuition.

## Next steps

Next: [Dates and panel data](08_panels_and_time.ipynb). See the [pandas groupby guide](https://pandas.pydata.org/docs/user_guide/groupby.html).

[Report a content error](https://github.com/ningzhanglab/Python-for-Beginners/issues/new?template=content_error.md).